In [1]:
import json
import pandas as pd

In [345]:
results_path = '../src/experiments/test4/res_results.csv' #to change
results = pd.read_csv(results_path, parse_dates=['strategy_last_date'])
results['strategy_last_date'] = results['strategy_last_date'].fillna(pd.Timestamp(2100, 1, 1))

In [346]:
def get_stats(df, script_name_filter=None):
    if script_name_filter is not None:
        df = df[df['scriptName'].apply(lambda x: script_name_filter(x))]
    stopped = len(df[df['prct_run'] < 1])/len(df)
    recall = len(df[df['roi_mdd'] <= df['roi_mdd_stopped']])/len(df)
    precision = len(df[df['roi_mdd'] < df['roi_mdd_stopped']])/len(df)
    stopped_roi_mdd_better = len(df[(df['prct_run'] < 1) & (df['roi_mdd'] < df['roi_mdd_stopped'])])/len(df)
    stopped_roi_mdd_worse = len(df[(df['prct_run'] < 1) & (df['roi_mdd'] > df['roi_mdd_stopped'])])/len(df)
    f1 = 2 * (precision * recall) / (precision + recall)
    return len(df), stopped, recall, precision, f1, stopped_roi_mdd_better, stopped_roi_mdd_worse

In [347]:
from itertools import chain, combinations

def powerset(iterable):
    s = list(iterable)  # allows duplicate elements
    return chain.from_iterable(combinations(s, r) for r in range(1, 3))

In [348]:
final_results = []
for combo in powerset(results['criteria'].unique()):
    local_list = []
    combo_list = list(combo)
    for criteria in combo:
        local_list.append(results[results['criteria'] == criteria].reset_index().drop(columns=['index']))
    local_df = pd.concat(local_list, axis=1, keys=list(combo)) if len(local_list) else None
    if local_df is not None:
        idxmin = local_df.loc[idx[:], idx[:, 'strategy_last_date']].idxmin(axis=1)
        for i in range(len(local_df)):
            least_date_combo = idxmin.iloc[i][0]
            metrics = list(local_df.loc[i, idx[least_date_combo, ['scriptName', 'timeframe', 'instrument', 'prct_run', 'roi_mdd', 'roi_mdd_stopped']]].values)
            metrics.append(combo_list)
            final_results.append(metrics)

In [349]:
final_results_df = pd.DataFrame(final_results, columns=['scriptName', 'timeframe', 'instrument', 'prct_run', 'roi_mdd', 'roi_mdd_stopped', 'combo_criteria_list'])

In [350]:
final_results_df['combo_criteria_list'] = final_results_df['combo_criteria_list'].apply(lambda x: json.dumps(x))

In [376]:
metrics = []
def mean_type(x):
    return 'Mean' in x or 'mean' in x
def trend_type(x):
    return 'Trend' in x or 'trend' in x

for i, group in enumerate(final_results_df.groupby('combo_criteria_list')):
    metrics.append([group[0]] + [x for s in [list(get_stats(group[1], func)) for func in [None, trend_type, mean_type]] for x in s])

In [377]:
metrics_df = pd.DataFrame(metrics, columns=["combo_criteria", "num", "stopped", "recall", "precision", "f1", "stopped_roi_mdd_better", "stopped_roi_mdd_worse",
                                            "num_trend", "stopped_trend", "recall_trend", "precision_trend", "f1_trend", "stopped_roi_mdd_better_trend", "stopped_roi_mdd_worse_trend",
                                            "num_mean", "stopped_mean", "recall_mean", "precision_mean", "f1_mean", "stopped_roi_mdd_better_mean", "stopped_roi_mdd_worse_mean"])

In [378]:
metrics_df.sort_values(by=['f1_trend'], axis=0).iloc[-1]["combo_criteria"]

'["trades_profit_months_stop_criteria_months:1,offset_months:6,p:0.1", "trades_daily_profit_stop_criteria_p:0.05,window:30"]'

In [379]:
metrics_df.sort_values(by=['f1_mean'], axis=0).iloc[-1]["combo_criteria"]

'["trades_profit_months_stop_criteria_months:2,offset_months:6,p:0.1", "trades_duration_stop_criteria_p:0.1,window:30"]'

In [375]:
metrics_df

,combo_criteria,num,stopped,recall,precision,f1,stopped_roi_mdd_better,stopped_roi_mdd_worse,num_trend,stopped_trend,...,f1_trend,stopped_roi_mdd_better_trend,stopped_roi_mdd_worse_trend,num_mean,stopped_mean,recall_mean,precision_mean,f1_mean,stopped_roi_mdd_better_mean,stopped_roi_mdd_worse_mean
0,"[""trades_daily_distribution_parameters_stop_cr...",146,0.602740,0.417808,0.020548,0.039170,0.020548,0.582192,112,0.589286,...,0.034286,0.017857,0.571429,34,0.647059,0.382353,0.029412,0.054622,0.029412,0.617647
1,"[""trades_daily_profit_stop_criteria_p:0.05,win...",146,0.890411,0.301370,0.191781,0.234399,0.191781,0.698630,112,0.883929,...,0.222372,0.178571,0.705357,34,0.911765,0.323529,0.235294,0.272446,0.235294,0.676471
2,"[""trades_daily_profit_stop_criteria_p:0.05,win...",146,0.835616,0.363014,0.198630,0.256766,0.198630,0.636986,112,0.857143,...,0.239224,0.187500,0.669643,34,0.764706,0.470588,0.235294,0.313725,0.235294,0.529412
3,"[""trades_daily_profit_stop_criteria_p:0.05,win...",146,0.828767,0.369863,0.198630,0.258458,0.198630,0.630137,112,0.848214,...,0.241525,0.187500,0.660714,34,0.764706,0.470588,0.235294,0.313725,0.235294,0.529412
4,"[""trades_daily_profit_stop_criteria_p:0.05,win...",146,0.828767,0.369863,0.198630,0.258458,0.198630,0.630137,112,0.848214,...,0.241525,0.187500,0.660714,34,0.764706,0.470588,0.235294,0.313725,0.235294,0.529412
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401,"[""trades_profit_stop_criteria_p:0.1,window:40""...",146,0.376712,0.808219,0.184932,0.300992,0.184932,0.191781,112,0.428571,...,0.324838,0.205357,0.223214,34,0.205882,0.911765,0.117647,0.208403,0.117647,0.088235
402,"[""trades_profit_stop_criteria_p:0.1,window:40""...",146,0.458904,0.760274,0.219178,0.340262,0.219178,0.239726,112,0.508929,...,0.340136,0.223214,0.285714,34,0.294118,0.911765,0.205882,0.335913,0.205882,0.088235
403,"[""trades_profit_stop_criteria_p:0.1,window:40""...",146,0.383562,0.801370,0.184932,0.300514,0.184932,0.198630,112,0.437500,...,0.312083,0.196429,0.241071,34,0.205882,0.941176,0.147059,0.254372,0.147059,0.058824
404,"[""trades_profit_stop_criteria_p:0.1,window:40""...",146,0.321918,0.828767,0.150685,0.255005,0.150685,0.171233,112,0.366071,...,0.267356,0.160714,0.205357,34,0.176471,0.941176,0.117647,0.209150,0.117647,0.058824
